In [4]:
import sys,os
sys.path.append('..')
sys.path.append(os.path.expanduser('~/github/lltk'))
import lltk
from largeliterarymodels import *
from largeliterarymodels.tasks.classify_genre import *
from tqdm import tqdm
from largeliterarymodels.llm import GEMINI_PRO, GEMINI_FLASH, CLAUDE_OPUS
import random
MODEL = CLAUDE_OPUS

In [5]:
C=lltk.load('arc_fiction')
# C.meta

In [6]:
df = C.meta.copy()
df['genre_raw'] = df['genre_raw'].fillna('')
# df=df.query('1475<=year<1700')
df = df.query('1700<=year<1800')
df = df[(df.genre_raw=='') | (df.genre_raw=='Fiction')]
len(df)

3111

In [7]:
n_max=1000
texts = [C.text(id) for id in tqdm(df.index)]
texts = random.sample(texts, min(len(texts), n_max))
len(texts)

100%|██████████| 3111/3111 [00:01<00:00, 3054.30it/s]


1000

In [8]:
texts[0].meta

{'_id': '_arc_fiction/mdp/39015055067113',
 '_corpus': 'arc_fiction',
 'id': 'mdp/39015055067113',
 'author': 'Burnet, Thomas,',
 'author_norm': 'burnet',
 'corpus': 'hathi_englit',
 'genre': 'Fiction',
 'genre_raw': 'Fiction',
 'path_freqs': 'hathi/freqs/mdp/39015055067113.json',
 'title': 'A second tale of a tub;',
 'title_norm': 'a second tale of a tub',
 'year': np.int32(1715)}

In [9]:
prompt=format_lltk_text_for_classification(texts[0])
print(prompt)

Title: A second tale of a tub;
Author: Burnet
Year: 1700-1800


In [10]:
task = GenreTask()

In [11]:
task.run(prompt, model=MODEL)

GenreClassification(genre='Essay', genre_raw='', author_first_name='Thomas', is_translated=False, translated_from='', year_estimated=1715, confidence=0.72, reasoning="This appears to be a satirical political pamphlet by Thomas Burnet (the younger), playing on Swift's 'A Tale of a Tub.' Works with this title from this period are typically political satire or polemical essays rather than sustained prose fiction.", notes="Thomas Burnet (c.1694–1753), son of the bishop and historian Gilbert Burnet, wrote several satirical pamphlets. 'A Second Tale of a Tub' (c.1715) is a political satire that alludes to Swift's famous 'A Tale of a Tub' (1704). Despite the word 'tale' in the title, it is a satirical essay or pamphlet rather than a work of prose fiction. It engages with contemporary political and ecclesiastical controversies.")

In [12]:
prompts = [format_lltk_text_for_classification(t) for t in texts]

results = task.map(prompts, model=MODEL)

Extracting GenreClassification (claude-opus-4-6): 100%|██████████| 969/969 [24:29<00:00,  1.52s/it]


In [29]:
assert len(results) == len(texts) == len(prompts)

In [30]:
r=results[-1]

In [31]:
r.model_dump()

{'genre': 'Fiction',
 'genre_raw': 'Imaginary voyage',
 'author_first_name': 'Savinien',
 'is_translated': True,
 'translated_from': 'French',
 'year_estimated': 1657,
 'confidence': 0.97,
 'reasoning': "Cyrano de Bergerac's 'Voyage to the Moon' (L'Autre Monde: ou les États et Empires de la Lune) is a classic French imaginary voyage, a pioneering work of proto-science fiction, translated into English.",
 'notes': "Originally published posthumously in French as 'Histoire comique des États et Empires de la Lune' (1657), this work by Savinien de Cyrano de Bergerac is one of the earliest examples of science fiction. The narrator travels to the moon and encounters its inhabitants, providing a vehicle for philosophical satire on religion, science, and society. It was followed by a companion volume about the sun. Multiple English translations appeared in the 18th century."}

In [32]:
results[-1], texts[-1]

(GenreClassification(genre='Fiction', genre_raw='Imaginary voyage', author_first_name='Savinien', is_translated=True, translated_from='French', year_estimated=1657, confidence=0.97, reasoning="Cyrano de Bergerac's 'Voyage to the Moon' (L'Autre Monde: ou les États et Empires de la Lune) is a classic French imaginary voyage, a pioneering work of proto-science fiction, translated into English.", notes="Originally published posthumously in French as 'Histoire comique des États et Empires de la Lune' (1657), this work by Savinien de Cyrano de Bergerac is one of the earliest examples of science fiction. The narrator travels to the moon and encounters its inhabitants, providing a vehicle for philosophical satire on religion, science, and society. It was followed by a companion volume about the sun. Multiple English translations appeared in the 18th century."),
 Bergerac, A Voyage To The Moon (1754) [_arc_fiction/LitAndLang1/0001000100])

In [33]:
import pandas as pd
ld = [{**t.meta, **{'llm_'+k:v for k,v in r.model_dump().items()}} for t,r in zip(texts,results)]
odf = pd.DataFrame(ld).fillna('')
# odf

In [34]:
odf.llm_genre_raw.value_counts().head(10)

llm_genre_raw
                      237
Novel                 142
Novel, sentimental     93
Chapbook               74
Tale                   42
Tale, oriental         40
Novel, epistolary      33
Fiction, satire        30
Novel, amatory         22
Novel, picaresque      19
Name: count, dtype: int64

In [35]:
odf.columns

Index(['_id', '_corpus', 'id', 'author', 'author_norm', 'corpus', 'genre',
       'genre_raw', 'path_freqs', 'title', 'title_norm', 'year', 'llm_genre',
       'llm_genre_raw', 'llm_author_first_name', 'llm_is_translated',
       'llm_translated_from', 'llm_year_estimated', 'llm_confidence',
       'llm_reasoning', 'llm_notes', 'is_translated', 'genre_source',
       'genre_source_reason', 'notes'],
      dtype='object')

In [36]:
correct_odf = odf[
    (odf.year == odf.llm_year_estimated)
    & odf.apply(lambda r: bool((r.author in {'Unknown','Anon','Anonymous',''}) or r.llm_author_first_name) and r.llm_author_first_name in r.author, axis=1)
    # & (odf.llm_confidence>=0.9)
].fillna('')
correct_odf['llm_genre_source_reason'] = correct_odf['llm_reasoning']
correct_odf['llm_llm_confidence'] = correct_odf['llm_confidence']
correct_odf[['id','corpus','author','title','year'] + [c for c in correct_odf if c.startswith('llm_')]]

,id,corpus,author,title,year,llm_genre,llm_genre_raw,llm_author_first_name,llm_is_translated,llm_translated_from,llm_year_estimated,llm_confidence,llm_reasoning,llm_notes,llm_genre_source_reason,llm_llm_confidence
0,mdp/39015055067113,hathi_englit,"Burnet, Thomas,",A second tale of a tub;,1715.0,Essay,,Thomas,False,,1715,0.72,This appears to be a satirical political pamph...,"Thomas Burnet (c.1694–1753), son of the bishop...",This appears to be a satirical political pamph...,0.72
12,HistAndGeo/0739200300,ecco,,"The whole life, and conversation, birth and pa...",1707.0,Fiction,Criminal biography,,False,,1707,0.72,This is a criminal biography/pamphlet recounti...,This is a typical early 18th-century criminal ...,This is a criminal biography/pamphlet recounti...,0.72
18,LitAndLang1/0250600500,ecco,,The Lancashire collier girl. A true story,1795.0,Fiction,"Novel, sentimental",,False,,1795,0.72,"Despite the subtitle 'A true story,' works wit...",The Lancashire Collier Girl: A True Story is a...,"Despite the subtitle 'A true story,' works wit...",0.72
19,LL1560,litlab,"Williams, Helen Maria",Edwin and Eltruda. A legendary tale. By a youn...,1782.0,Poetry,,Helen Maria,False,,1782,0.90,Helen Maria Williams's 'Edwin and Eltruda' (17...,Edwin and Eltruda: A Legendary Tale (1782) was...,Helen Maria Williams's 'Edwin and Eltruda' (17...,0.90
20,LL1221,litlab,Unknown,"The Birmingham counterfeit; or, invisible spec...",1772.0,Fiction,"Novel, sentimental",,False,,1772,0.85,The title explicitly identifies itself as 'a s...,"The Birmingham Counterfeit; or, Invisible Spec...",The title explicitly identifies itself as 'a s...,0.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
978,K061006.000,tedjdh,"Defoe, Daniel, 1661?-1731",THE SHORTEST-WAY WITH THE DISSENTERS: OR PROPO...,1702.0,Essay,,Daniel,False,,1702,0.97,This is Daniel Defoe's famous satirical pamphl...,The Shortest-Way with the Dissenters (1702) by...,This is Daniel Defoe's famous satirical pamphl...,0.97
983,1764/Fiction/1468,bpo,,"A CHAPTER read, as if from the Bible, by a Gen...",1764.0,Essay,,,False,,1764,0.75,This is a satirical pamphlet written in biblic...,This pamphlet is almost certainly related to t...,This is a satirical pamphlet written in biblic...,0.75
988,HistAndGeo/0091100400,ecco,,"The true and remarkable trial, condemnation, a...",1793.0,History,,,False,,1793,0.90,This is a factual account of the trial and exe...,Louis XVI was executed by guillotine on Januar...,This is a factual account of the trial and exe...,0.90
993,LL1358,litlab,Unknown,The life and villainous actions of that notori...,1725.0,Fiction,Criminal biography,,False,,1725,0.88,"This is a criminal biography of Jonathan Wild,...",Jonathan Wild (c.1683–1725) was a famous Londo...,"This is a criminal biography of Jonathan Wild,...",0.88


In [37]:
len(correct_odf)

209

In [38]:
# correct_odf

In [39]:
# task.stash.df

In [40]:
def get_record(row):
    o = {}
    o['_id'] = f'_{row.corpus}/{row.id}'
    o['genre_source'] = f'llm:{MODEL}'
    for k in ['genre','genre_raw','is_translated','translated_from','genre_source_reason','notes', 'llm_confidence']:
        val = row['llm_'+k]
        if val!='':
            o[k] = val
    return o

In [41]:
records = [get_record(row) for i,row in correct_odf.iterrows()]
records

[{'_id': '_hathi_englit/mdp/39015055067113',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Essay',
  'is_translated': False,
  'genre_source_reason': "This appears to be a satirical political pamphlet by Thomas Burnet (the younger), playing on Swift's 'A Tale of a Tub.' Works with this title from this period are typically political satire or polemical essays rather than sustained prose fiction.",
  'notes': "Thomas Burnet (c.1694–1753), son of the bishop and historian Gilbert Burnet, wrote several satirical pamphlets. 'A Second Tale of a Tub' (c.1715) is a political satire that alludes to Swift's famous 'A Tale of a Tub' (1704). Despite the word 'tale' in the title, it is a satirical essay or pamphlet rather than a work of prose fiction. It engages with contemporary political and ecclesiastical controversies.",
  'llm_confidence': 0.72},
 {'_id': '_ecco/HistAndGeo/0739200300',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Fiction',
  'genre_raw': 'Criminal biography',
  '

In [43]:
pd.DataFrame(records).fillna('')

,_id,genre_source,genre,is_translated,genre_source_reason,notes,llm_confidence,genre_raw,translated_from
0,_hathi_englit/mdp/39015055067113,llm:claude-opus-4-6,Essay,False,This appears to be a satirical political pamph...,"Thomas Burnet (c.1694–1753), son of the bishop...",0.72,,
1,_ecco/HistAndGeo/0739200300,llm:claude-opus-4-6,Fiction,False,This is a criminal biography/pamphlet recounti...,This is a typical early 18th-century criminal ...,0.72,Criminal biography,
2,_ecco/LitAndLang1/0250600500,llm:claude-opus-4-6,Fiction,False,"Despite the subtitle 'A true story,' works wit...",The Lancashire Collier Girl: A True Story is a...,0.72,"Novel, sentimental",
3,_litlab/LL1560,llm:claude-opus-4-6,Poetry,False,Helen Maria Williams's 'Edwin and Eltruda' (17...,Edwin and Eltruda: A Legendary Tale (1782) was...,0.90,,
4,_litlab/LL1221,llm:claude-opus-4-6,Fiction,False,The title explicitly identifies itself as 'a s...,"The Birmingham Counterfeit; or, Invisible Spec...",0.85,"Novel, sentimental",
...,...,...,...,...,...,...,...,...,...
204,_tedjdh/K061006.000,llm:claude-opus-4-6,Essay,False,This is Daniel Defoe's famous satirical pamphl...,The Shortest-Way with the Dissenters (1702) by...,0.97,,
205,_bpo/1764/Fiction/1468,llm:claude-opus-4-6,Essay,False,This is a satirical pamphlet written in biblic...,This pamphlet is almost certainly related to t...,0.75,,
206,_ecco/HistAndGeo/0091100400,llm:claude-opus-4-6,History,False,This is a factual account of the trial and exe...,Louis XVI was executed by guillotine on Januar...,0.90,,
207,_litlab/LL1358,llm:claude-opus-4-6,Fiction,False,"This is a criminal biography of Jonathan Wild,...",Jonathan Wild (c.1683–1725) was a famous Londo...,0.88,Criminal biography,


In [44]:
C.propagate_from(records, dry_run=True)

Would write 209 annotation entries:
  litlab: 75
  ecco: 57
  hathi_englit: 37
  tedjdh: 20
  ecco_tcp: 5
  earlyprint: 4
  bpo: 4
  clmet: 3
  chadwyck: 3
  gale_amfic: 1


[{'_id': '_hathi_englit/mdp/39015055067113',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Essay',
  'is_translated': False,
  'genre_source_reason': "This appears to be a satirical political pamphlet by Thomas Burnet (the younger), playing on Swift's 'A Tale of a Tub.' Works with this title from this period are typically political satire or polemical essays rather than sustained prose fiction.",
  'notes': "Thomas Burnet (c.1694–1753), son of the bishop and historian Gilbert Burnet, wrote several satirical pamphlets. 'A Second Tale of a Tub' (c.1715) is a political satire that alludes to Swift's famous 'A Tale of a Tub' (1704). Despite the word 'tale' in the title, it is a satirical essay or pamphlet rather than a work of prose fiction. It engages with contemporary political and ecclesiastical controversies.",
  'llm_confidence': 0.72},
 {'_id': '_ecco/HistAndGeo/0739200300',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Fiction',
  'is_translated': False,
  'genre_sourc

In [45]:
C.propagate_from(records)

Wrote 209 annotation entries from external
  litlab: 75
  ecco: 57
  hathi_englit: 37
  tedjdh: 20
  ecco_tcp: 5
  earlyprint: 4
  bpo: 4
  clmet: 3
  chadwyck: 3
  gale_amfic: 1


[{'_id': '_hathi_englit/mdp/39015055067113',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Essay',
  'is_translated': False,
  'genre_source_reason': "This appears to be a satirical political pamphlet by Thomas Burnet (the younger), playing on Swift's 'A Tale of a Tub.' Works with this title from this period are typically political satire or polemical essays rather than sustained prose fiction.",
  'notes': "Thomas Burnet (c.1694–1753), son of the bishop and historian Gilbert Burnet, wrote several satirical pamphlets. 'A Second Tale of a Tub' (c.1715) is a political satire that alludes to Swift's famous 'A Tale of a Tub' (1704). Despite the word 'tale' in the title, it is a satirical essay or pamphlet rather than a work of prose fiction. It engages with contemporary political and ecclesiastical controversies.",
  'llm_confidence': 0.72},
 {'_id': '_ecco/HistAndGeo/0739200300',
  'genre_source': 'llm:claude-opus-4-6',
  'genre': 'Fiction',
  'is_translated': False,
  'genre_sourc